In [50]:
import polars as pl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments,
    BertForSequenceClassification,
    BertTokenizer
)
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from huggingface_hub import snapshot_download
import warnings
import math


warnings.filterwarnings('ignore')

In [51]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)


HISTORY_ACTIONS = ["view", "click"]  
TARGET_ACTIONS = ["clickout", "like"]  
ALL_ACTIONS = ["view", "click", "clickout", "like"]

MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000
TOP_K = 15
BATCH_SIZE_TRAIN = 5000      
MAX_SEQ_LEN = 30             
MIN_SEQ_LEN = 3
TOP_N_ITEMS = 5000          
SAMPLE_USERS = 5000         
SAMPLE_RATIO = 0.1 

action_weights = {
    "view": 1.0,
    "click": 2.0,
    "like": 4.0,
    "clickout": 8.0  
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [52]:
snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

Fetching ... files: 0it [00:00, ?it/s]

'/kaggle/working'

In [53]:
events_path = "/kaggle/working/dataset/small/marketplace/events"
items_path = "/kaggle/working/dataset/small/marketplace/items.pq"

events_lf = pl.scan_parquet(events_path).select([
    "timestamp", "user_id", "item_id", "action_type", "subdomain", "os"
])

In [54]:
items_lf = pl.scan_parquet(items_path).select([
    "item_id", "brand_id", "category", "subcategory", "price"
])

In [55]:
items_lf = items_lf.drop_nulls(subset=["price"]).with_columns([
    pl.col("category").fill_null("unknown"),
    pl.col("subcategory").fill_null("unknown"),
    pl.col("brand_id").fill_null(-1),
])


In [56]:
events_lf = pl.scan_parquet(events_path).select([
    "timestamp", "user_id", "item_id", "action_type", "subdomain", "os"
])

events_lf = events_lf.with_columns([
    (pl.col("timestamp") / 1_000_000).cast(pl.Int64).alias("day")
])

events_lf = events_lf.filter(pl.col("action_type").is_in(ALL_ACTIONS))

days = (
    events_lf
    .select(pl.col("day").unique())
    .sort("day")
    .collect()
)["day"].to_list()

val_threshold_idx = int(len(days) * 0.7)
test_threshold_idx = int(len(days) * 0.8)
val_threshold = days[val_threshold_idx]
test_threshold = days[test_threshold_idx]


train_events_lf = events_lf.filter(pl.col("day") < val_threshold)
val_events_lf = events_lf.filter((pl.col("day") >= val_threshold) & (pl.col("day") < test_threshold))
test_events_lf = events_lf.filter(pl.col("day") >= test_threshold)

history_events_lf = train_events_lf.filter(pl.col("action_type").is_in(HISTORY_ACTIONS))

top_items = (
    history_events_lf
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_N_ITEMS)
    .select("item_id")
    .collect()
)["item_id"].to_list()

In [57]:
train_history = (
    train_events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(HISTORY_ACTIONS))
    .collect(streaming=True)
)

train_targets = (
    train_events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(TARGET_ACTIONS))
    .collect(streaming=True)
)

val_history = (
    val_events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(HISTORY_ACTIONS))
    .collect(streaming=True)
)

val_targets = (
    val_events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(TARGET_ACTIONS))
    .collect(streaming=True)
)

val_events = val_events_lf.collect(streaming=True)

test_events = (
    test_events_lf
    .filter(pl.col("action_type").is_in(ALL_ACTIONS))
    .collect(streaming=True)
)

In [58]:
train_targets = train_targets.with_columns([
    pl.col("action_type").replace(action_weights).alias("target_weight")
])

val_targets = val_targets.with_columns([
    pl.col("action_type").replace(action_weights).alias("target_weight")
])

val_events = val_events.with_columns([
    pl.col("action_type").replace(action_weights).alias("target_weight")
])

test_events = test_events.with_columns([
    pl.col("action_type").replace(action_weights).alias("target_weight")
])

train_events = train_history

In [59]:
users_with_history = set(train_events["user_id"].unique().to_list())
users_with_targets = set(train_targets["user_id"].unique().to_list())
users_with_val_targets = set(val_targets["user_id"].unique().to_list())
users_with_test = set(test_events["user_id"].unique().to_list())

common_users = users_with_history & users_with_targets & users_with_val_targets & users_with_test
common_users = list(common_users)

if len(common_users) > SAMPLE_USERS:
    np.random.seed(SEED)
    sampled_users = np.random.choice(common_users, SAMPLE_USERS, replace=False)
else:
    sampled_users = common_users

train_events = train_events.filter(pl.col("user_id").is_in(sampled_users))
train_targets = train_targets.filter(pl.col("user_id").is_in(sampled_users))
val_targets = val_targets.filter(pl.col("user_id").is_in(sampled_users))
val_events = val_events.filter(pl.col("user_id").is_in(sampled_users))
test_events = test_events.filter(pl.col("user_id").is_in(sampled_users))


In [60]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
) -> pl.DataFrame:
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Vectorized implementation using polars operations instead of per-user loops.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively.
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    _ITEM = "__item"
    _RANK = "__rank"

    truth = (
        user_based_df.select("user_id", true_items_col, relevancy_col)
        .explode(true_items_col, relevancy_col)
        .group_by("user_id", true_items_col)
        .agg(pl.col(relevancy_col).max())
        .rename({true_items_col: _ITEM})
    )

    preds = (
        user_based_df.select("user_id", predicted_items_col, predicted_score_col)
        .explode(predicted_items_col, predicted_score_col)
        .rename({predicted_items_col: _ITEM})
    )

    preds_rel = preds.join(truth, on=["user_id", _ITEM], how="left").fill_null(0)

    gain_expr = (pl.col(relevancy_col) / (pl.col(_RANK) + 1).cast(pl.Float64).log(2)).sum()

    dcg = (
        preds_rel.with_columns(
            pl.col(predicted_score_col)
            .rank("ordinal", descending=True)
            .over("user_id")
            .alias(_RANK)
        )
        .filter(pl.col(_RANK) <= top_k)
        .group_by("user_id")
        .agg(gain_expr.alias("dcg"))
    )

    idcg = (
        truth.with_columns(
            pl.col(relevancy_col).rank("ordinal", descending=True).over("user_id").alias(_RANK)
        )
        .filter(pl.col(_RANK) <= top_k)
        .group_by("user_id")
        .agg(gain_expr.alias("idcg"))
    )

    return (
        dcg.join(idcg, on="user_id", how="left")
        .with_columns(
            pl.when(pl.col("idcg").is_null() | (pl.col("idcg") == 0))
            .then(0.0)
            .otherwise(pl.col("dcg") / pl.col("idcg"))
            .alias("ndcg")
        )
        .select("user_id", "ndcg")
    )


def _ndcg_at_k_loop(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
) -> pl.DataFrame:
    """Исходная реализация ndcg_at_k через цикл по пользователям (для тестов)."""
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(pl.col("relevancy").max())
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [61]:
def precision_recall_at_k(
    df: pl.DataFrame,
    predicted_items_col: str,
    true_items_col: str,
    top_k: int = 15,
) -> tuple[float, float]:
    """
    Вычисляет средние Precision@k и Recall@k по всем пользователям.
    """
    top_k_preds = pl.col(predicted_items_col).list.head(top_k)
    hits_expr = top_k_preds.list.set_intersection(pl.col(true_items_col)).list.len()

    metrics = df.select(
        (hits_expr / top_k).mean().alias("precision"),
        (hits_expr / pl.col(true_items_col).list.len()).mean().alias("recall"),
    )
    return metrics["precision"].item(), metrics["recall"].item()

In [62]:
item_to_idx = {item: idx + 1 for idx, item in enumerate(top_items)} 
idx_to_item = {idx + 1: item for idx, item in enumerate(top_items)}  
num_items = len(top_items) + 1 

In [63]:
class RankingDataset(Dataset):
    def __init__(self, history_df, targets_df=None, item_to_idx=None, max_seq_len=30, min_seq_len=3, mode='train'):
        """
        mode: 'train' – каждый таргет становится отдельным примером, история берётся ДО этого таргета.
              'test'  – для каждого пользователя строится одна последовательность из всей истории (train).
        Для mode='train' обязательны history_df и targets_df (с колонками user_id, item_id, timestamp).
        Для mode='test'  используется только history_df.
        """
        self.max_seq_len = max_seq_len
        self.min_seq_len = min_seq_len
        self.item_to_idx = item_to_idx
        self.mode = mode
        self.data = []

        # Сортируем историю и группируем по пользователю
        history_df = history_df.select(['user_id', 'item_id', 'timestamp']).sort(['user_id', 'timestamp'])
        history_groups = history_df.group_by('user_id').agg([
            pl.col('item_id').alias('hist_items'),
            pl.col('timestamp').alias('hist_times')
        ])
        self.hist_dict = {}
        for row in history_groups.iter_rows(named=True):
            self.hist_dict[row['user_id']] = (list(row['hist_items']), list(row['hist_times']))

        if mode == 'train':
            if targets_df is None:
                raise ValueError("For training mode, targets_df must be provided.")
            targets_df = targets_df.select(['user_id', 'item_id', 'timestamp']).sort(['user_id', 'timestamp'])
            for row in targets_df.iter_rows(named=True):
                user_id = row['user_id']
                target_item = row['item_id']
                target_time = row['timestamp']
                if user_id not in self.hist_dict:
                    continue
                hist_items, hist_times = self.hist_dict[user_id]
                # Фильтруем историю строго до target_time
                filtered = [(item, t) for item, t in zip(hist_items, hist_times) if t < target_time]
                if len(filtered) < self.min_seq_len:
                    continue
                filtered = filtered[-self.max_seq_len:]
                item_indices = [self.item_to_idx.get(item, 0) for item, _ in filtered]
                seq_len = len(item_indices)
                padded_seq = item_indices + [0] * (self.max_seq_len - seq_len)
                attention_mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)
                target_idx = self.item_to_idx.get(target_item, 0)
                if target_idx == 0:
                    continue
                self.data.append({
                    'user_id': user_id,
                    'input_ids': padded_seq,
                    'attention_mask': attention_mask,
                    'target': target_idx
                })
        else:  # test mode – используем всю train-историю для каждого пользователя
            for user_id, (hist_items, _) in self.hist_dict.items():
                if len(hist_items) < self.min_seq_len:
                    continue
                hist_items = hist_items[-self.max_seq_len:]
                item_indices = [self.item_to_idx.get(item, 0) for item in hist_items]
                seq_len = len(item_indices)
                padded_seq = item_indices + [0] * (self.max_seq_len - seq_len)
                attention_mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)
                self.data.append({
                    'user_id': user_id,
                    'input_ids': padded_seq,
                    'attention_mask': attention_mask
                })
        print(f"Dataset created. Mode: {mode}, samples: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        result = {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "user_id": torch.tensor(item["user_id"], dtype=torch.long)
        }
        if "target" in item:
            result["target"] = torch.tensor(item["target"], dtype=torch.long)
        return result

In [64]:
BATCH_SIZE = 64

train_dataset = RankingDataset(
    history_df=train_events,         
    targets_df=train_targets,      
    item_to_idx=item_to_idx,
    max_seq_len=MAX_SEQ_LEN,
    min_seq_len=MIN_SEQ_LEN,
    mode='train'
)

val_dataset = RankingDataset(
    history_df=train_events,         
    targets_df=None,
    item_to_idx=item_to_idx,
    max_seq_len=MAX_SEQ_LEN,
    min_seq_len=MIN_SEQ_LEN,
    mode='test'                   
)

test_dataset = RankingDataset(
    history_df=train_events,         
    targets_df=None,
    item_to_idx=item_to_idx,
    max_seq_len=MAX_SEQ_LEN,
    min_seq_len=MIN_SEQ_LEN,
    mode='test'
)

val_true_targets = (
    val_events
    .filter(pl.col("action_type").is_in(TARGET_ACTIONS))
    .filter(pl.col("item_id").is_in(top_items))
)

test_true_targets = (
    test_events
    .filter(pl.col("action_type").is_in(TARGET_ACTIONS))
    .filter(pl.col("item_id").is_in(top_items))
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)


Dataset created. Mode: train, samples: 13458
Dataset created. Mode: test, samples: 4015
Dataset created. Mode: test, samples: 4015


In [65]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [66]:
class RankingTransformer(nn.Module):
    """Модель для ранжирования: возвращает эмбеддинг пользователя"""
    
    def __init__(self, num_items, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=512, dropout=0.3, max_seq_len=50):
        super().__init__()
        
        self.d_model = d_model
        self.num_items = num_items
        
        self.item_embedding = nn.Embedding(num_items, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        
        x = self.item_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        seq_lengths = attention_mask.sum(dim=1) - 1
        seq_lengths = torch.clamp(seq_lengths, min=0)
        
        causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1).to(x.device)
        src_key_padding_mask = (attention_mask == 0)
        transformer_out = self.transformer(x, mask=causal_mask, src_key_padding_mask=src_key_padding_mask)
        
        batch_indices = torch.arange(batch_size, device=input_ids.device)
        user_embedding = transformer_out[batch_indices, seq_lengths]
        
        return user_embedding
    
    def predict_scores(self, user_embedding, candidate_ids):
        candidate_embeddings = self.item_embedding(candidate_ids)
        scores = torch.bmm(candidate_embeddings, user_embedding.unsqueeze(-1)).squeeze(-1)
        return scores

In [67]:
model = RankingTransformer(
    num_items=num_items,
    d_model=128,
    nhead=8,
    num_layers=3,
    dim_feedforward=512,
    dropout=0.3,
    max_seq_len=MAX_SEQ_LEN
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [68]:
EPOCHS = 20
best_val_loss = float('inf')
train_losses = []
val_losses = []

patience_counter = 0
early_stopping_patience = 5

def evaluate_validation(model, dataloader, val_true_targets, idx_to_item, top_k=15):
    model.eval()
    all_predictions = []
    user_ids = []
    
    with torch.no_grad():
        all_item_ids = torch.arange(model.num_items, device=next(model.parameters()).device)
        item_embeddings = model.item_embedding(all_item_ids).cpu().numpy()
        
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            batch_user_ids = batch["user_id"].cpu().numpy()
            
            user_embeddings = model(input_ids, attention_mask).cpu().numpy()
            
            for i, (user_id, user_emb) in enumerate(zip(batch_user_ids, user_embeddings)):
                scores = np.dot(item_embeddings, user_emb)
                scores[0] = -np.inf
                
                top_k_indices = np.argsort(scores)[-top_k:][::-1]
                predicted_items = [idx_to_item.get(idx, idx) for idx in top_k_indices if idx != 0]
                
                all_predictions.append({
                    "user_id": int(user_id),
                    "predicted_items": predicted_items[:top_k]
                })
                user_ids.append(int(user_id))
    
    predictions_df = pl.DataFrame(all_predictions)
    
    true_targets_df = (
        val_true_targets
        .group_by("user_id")
        .agg(pl.col("item_id").alias("true_items"))
    )
    
    merged_df = predictions_df.join(true_targets_df, on="user_id", how="inner")
    if len(merged_df) == 0:
        return 0.0
    
    hit_rate_df = merged_df.with_columns(
        (pl.col("predicted_items").list.head(top_k)
         .list.set_intersection(pl.col("true_items"))
         .list.len() > 0).alias("hit")
    )
    return hit_rate_df["hit"].mean()

best_val_metric = -float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)
        
        optimizer.zero_grad()
        user_embedding = model(input_ids, attention_mask)
        batch_size = input_ids.size(0)
        all_item_ids_tensor = torch.arange(num_items, device=device).unsqueeze(0).expand(batch_size, -1)
        all_scores = model.predict_scores(user_embedding, all_item_ids_tensor)
        loss = criterion(all_scores, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    val_hit_rate = evaluate_validation(model, val_loader, val_true_targets, idx_to_item, TOP_K)
    val_losses.append(1.0 - val_hit_rate)
    scheduler.step(1.0 - val_hit_rate)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.6f}")
    print(f"  Val Hit Rate: {val_hit_rate:.6f}")
    print(f"  LR: {current_lr:.6f}")
    
    if val_hit_rate > best_val_metric:
        best_val_metric = val_hit_rate
        torch.save(model.state_dict(), "best_ranking_model.pt")
        patience_counter = 0
        print(f"  New best model! Hit Rate: {val_hit_rate:.6f}")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{early_stopping_patience})")
        if patience_counter >= early_stopping_patience:
            print(f"  Early stopping triggered at epoch {epoch+1}")
            break

Validation: 100%|██████████| 63/63 [00:03<00:00, 17.31it/s]



Epoch 1/20
  Train Loss: 7.257383
  Val Hit Rate: 0.188294
  LR: 0.001000
  New best model! Hit Rate: 0.188294


Validation: 100%|██████████| 63/63 [00:03<00:00, 17.34it/s]



Epoch 2/20
  Train Loss: 6.555065
  Val Hit Rate: 0.209963
  LR: 0.001000
  New best model! Hit Rate: 0.209963


Validation: 100%|██████████| 63/63 [00:03<00:00, 17.20it/s]


...


Epoch 5/20
  Train Loss: 4.901928
  Val Hit Rate: 0.174844
  LR: 0.001000
  No improvement (3/5)


Validation: 100%|██████████| 63/63 [00:03<00:00, 16.91it/s]



Epoch 6/20
  Train Loss: 4.372444
  Val Hit Rate: 0.156413
  LR: 0.000500
  No improvement (4/5)


Validation: 100%|██████████| 63/63 [00:03<00:00, 16.43it/s]


Epoch 7/20
  Train Loss: 3.614975
  Val Hit Rate: 0.133998
  LR: 0.000500
  No improvement (5/5)
  Early stopping triggered at epoch 7


In [69]:
model.load_state_dict(torch.load("best_ranking_model.pt"))
model.eval()

test_true_interactions = (
    test_events
    .filter(pl.col("action_type").is_in(TARGET_ACTIONS))
    .filter(pl.col("item_id").is_in(top_items))
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("target_weight").cast(pl.Float64).alias("relevancy")
    ])
)

def get_user_embeddings(model, dataloader):
    model.eval()
    user_embeddings = []
    user_ids = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing user embeddings"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            embeddings = model(input_ids, attention_mask).cpu().numpy()
            user_embeddings.append(embeddings)
            user_ids.extend(batch["user_id"].numpy())
    return np.vstack(user_embeddings), user_ids

print("Computing user embeddings...")
user_embeddings, user_ids = get_user_embeddings(model, test_loader)
print(f"Computed embeddings for {len(user_ids)} users")

all_item_ids = torch.arange(num_items, device=device).unsqueeze(0)
with torch.no_grad():
    all_item_embeddings = model.item_embedding(all_item_ids).squeeze(0).cpu().numpy()

TOP_K = 15
all_predictions = []
for i, (user_id, user_emb) in enumerate(tqdm(zip(user_ids, user_embeddings), total=len(user_ids), desc="Ranking items")):
    scores = np.dot(all_item_embeddings, user_emb)
    scores[0] = -np.inf
    top_k_indices = np.argsort(scores)[-TOP_K:][::-1]
    predicted_items = [idx_to_item.get(idx, idx) for idx in top_k_indices if idx != 0]
    predicted_scores = [float(scores[idx]) for idx in top_k_indices if idx != 0]
    all_predictions.append({
        "user_id": int(user_id),
        "predicted_items": predicted_items[:TOP_K],
        "predicted_scores": predicted_scores[:TOP_K]
    })

predictions_df = pl.DataFrame(all_predictions)
print(f"Predictions for {len(predictions_df)} users")

user_metrics_df = predictions_df.join(test_true_interactions, on="user_id", how="inner")
print(f"Users with both predictions and true interactions: {len(user_metrics_df)}")

if len(user_metrics_df) > 0:
    ndcg_per_user_df = ndcg_at_k(
        user_metrics_df, 
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=TOP_K
    )
    ndcg_score = ndcg_per_user_df["ndcg"].mean() if len(ndcg_per_user_df) > 0 else 0.0

    precision_score, recall_score = precision_recall_at_k(
        user_metrics_df, 
        predicted_items_col="predicted_items", 
        true_items_col="true_items", 
        top_k=TOP_K
    )

    hit_rate_df = user_metrics_df.with_columns(
        (pl.col("predicted_items").list.head(TOP_K)
         .list.set_intersection(pl.col("true_items"))
         .list.len() > 0).alias("hit")
    )
    hit_rate = hit_rate_df["hit"].mean()

    results = pl.DataFrame({
        "top_k": [TOP_K],
        "ndcg": [ndcg_score],
        "precision": [precision_score],
        "recall": [recall_score],
        "hit_rate": [hit_rate],
        "num_users": [len(user_metrics_df)],
        "num_items": [num_items - 1]
    })

    print("\n" + "="*80)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ")
    print("="*80)
    print(results)
    print("="*80)

    print(f"\n ДЕТАЛЬНЫЙ АНАЛИЗ:")
    print(f"  NDCG@{TOP_K}:      {ndcg_score:.6f}")
    print(f"  Precision@{TOP_K}: {precision_score:.6f}")
    print(f"  Recall@{TOP_K}:    {recall_score:.6f}")
    print(f"  Hit Rate@{TOP_K}:  {hit_rate:.6f}")
    print(f"  Users: {len(user_metrics_df)}")
    print(f"  Items: {num_items - 1}")
    print("="*80)
else:
    print("\n" + "="*80)
    print("Нет пользователей для оценки")
    print("="*80)

Computing user embeddings...


Computing user embeddings: 100%|██████████| 63/63 [00:01<00:00, 33.17it/s]


Computed embeddings for 4015 users


Ranking items: 100%|██████████| 4015/4015 [00:00<00:00, 4849.38it/s]


Predictions for 4015 users
Users with both predictions and true interactions: 1012

 РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ
shape: (1, 7)
┌───────┬──────────┬───────────┬──────────┬──────────┬───────────┬───────────┐
│ top_k ┆ ndcg     ┆ precision ┆ recall   ┆ hit_rate ┆ num_users ┆ num_items │
│ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---      ┆ ---       ┆ ---       │
│ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64      ┆ i64       ┆ i64       │
╞═══════╪══════════╪═══════════╪══════════╪══════════╪═══════════╪═══════════╡
│ 15    ┆ 0.106086 ┆ 0.021739  ┆ 0.164987 ┆ 0.289526 ┆ 1012      ┆ 5000      │
└───────┴──────────┴───────────┴──────────┴──────────┴───────────┴───────────┘

 ДЕТАЛЬНЫЙ АНАЛИЗ:
  NDCG@15:      0.106086
  Precision@15: 0.021739
  Recall@15:    0.164987
  Hit Rate@15:  0.289526
  Users: 1012
  Items: 5000
